# ML-02 — Mapping Lane 4 (CTR / Engagement Opportunity Scoring) onto the ML loop

This notebook does the framing work by hand first, then makes it real with a slice of the
starter data. Per the `framing-ml-problems` skill: **a model is never the goal, a better decision
is the goal** — so every code cell below exists to support one of the framing answers in the
markdown cells around it, not the other way around.

Runs on the real starter dataset (`data/raw/content_refresh_anonymized.csv`, 30,000 rows ×
44 columns, 32 clients) per the `flyrank-data` skill.

## 1. Task type

**Ranking / scoring.**

The question is *"which pages should a reviewer look at first?"* — that's the "which ones
first?" row in the `framing-ml-problems` task-type table, which maps directly to ranking/scoring,
not classification. There is no clean yes/no outcome to predict here (no "this page was reviewed
and it worked" label exists in the data), and there's no undiscovered grouping to find (position
tier is a known, predefined bucket, not a cluster to be discovered). The deliverable is an
**ordered list with a score attached to each row**, which is the definition of a ranking output.

## 2. Target / proxy

There is no observed ground-truth label for "should be reviewed" — nobody logged which pages a
human reviewer actually opened and fixed. So instead of manufacturing a fake classification label
out of the same features I'd use to rank (that would just teach a model my own rule, which the
framing skill explicitly warns against — *"the target must be observed, not defined"*), I predict
nothing and **score** instead:

```
gap_score = expected_ctr(position_tier) − actual_ctr
```

`expected_ctr(position_tier)` is the median CTR of *all other pages in that same position tier* —
a deterministic, leakage-safe baseline computed straight from the data, not a model prediction.
The full priority score (built in section 7) folds in impression volume (so the ranking isn't
dominated by noisy, low-traffic pages) and content freshness — using the dataset's own
`days_since_last_update` field, not a fabricated publish date — so a page that was just refreshed
isn't mistaken for a page whose title/meta has genuinely gone stale. None of these inputs — CTR,
tier, impressions, days since last update — depend on any future information, so the score is
safe to compute the day a page is evaluated.

## 3. Success metric

**Precision@K.**

A reviewer only has time to open a handful of pages in a week — say the top 20 in the queue. The
question that matters isn't "did we find every underperforming page" (recall) or "is the whole
30,000-row dataset labeled correctly" (accuracy) — it's **"of the top K pages we handed the
reviewer, how many turned out to be genuine, worthwhile gaps?"** That's precision@K, and it
matches the cost structure directly: a false positive at the top of the queue burns 10–20 minutes
of a reviewer's time on a page that didn't need it; a missed genuine gap (false negative) just
waits for next week's queue — it doesn't break anything. Optimizing for recall or overall accuracy
would reward a method that stuffs the queue with borderline/noisy cases just to avoid missing
one — the wrong trade-off for this decision.

## 4. Action the output supports

A FlyRank SEO/content reviewer opens the **top of the ranked queue** and, for each candidate,
sees why it's there (reason codes: high impressions, strong position, big CTR gap vs. tier, long
time since last update) and rewrites the title tag, meta description, or snippet structure — a
cheap, reversible content edit. The output is decision support, not an automated action: a human
reads the reason codes and decides whether to act, override, or skip.

## 5. Why ML/statistics beats a fixed rule here

A flat rule like `ctr < 0.5%` silently penalizes almost every page ranked below position 5, because
CTR mechanically falls as position gets worse — it has nothing to do with title/meta quality. The
right comparison — a page vs. *its own tier's* peers, adjusted further by how much traffic is at
stake and how long it's been since the page was actually touched — has too many interacting
factors to hand-write as a single if-statement, but a residual/gap calculation captures it cleanly:
it's exactly the kind of "real but tangled, multi-signal pattern" the `framing-ml-problems` skill
says is where a simple statistical model earns its place over a plain rule.

In [1]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"rows: {len(df):,}  |  columns: {df.shape[1]}  |  clients: {df.client_id.nunique()}")
df.head()


rows: 30,000  |  columns: 44  |  clients: 32


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 6. Unit of analysis

**One row = one content page (`content_id`), summarized over its trailing 90-day window.** Not
one row per query, per day, or per client — one page's aggregated visibility/click behavior. That
matches the decision: "which page should a reviewer open first" operates at the page level, so the
unit of analysis has to be the page level too.

The cell below filters down to the actual **decision surface**: pages that are visible (a real
position exists — recall `avg_position == 0` means "no data," not rank zero) and have enough
impressions to be worth a reviewer's time and statistically trustworthy at all. Everything past
this point in the pipeline only operates on this slice.

In [2]:
# Unit of analysis: one row = one visible content page (content_id), 90-day window
visible = df[(df.avg_position > 0) & (df.impressions_90d >= 500)].copy()

print(f"pages with real visibility (avg_position>0, impressions_90d>=500): "
      f"{len(visible):,} of {len(df):,} total ({len(visible)/len(df)*100:.1f}%)")

visible[["content_id", "client_id", "position_tier", "avg_position",
         "impressions_90d", "ctr", "days_since_last_update", "main_intent"]].head()


pages with real visibility (avg_position>0, impressions_90d>=500): 16,726 of 30,000 total (55.8%)


,content_id,client_id,position_tier,avg_position,impressions_90d,ctr,days_since_last_update,main_intent
0,content_304f48230142,client_f369cb89fc,striking,10.6,3803,0.76,20,transactional
1,content_a1fb4e703a9e,client_4e07408562,page_3_5,20.3,15320,0.05,25,informational
2,content_9aa793d4d895,client_7f2253d7e2,page_3_5,36.5,12581,0.09,20,informational
3,content_331d6c4de07b,client_19581e27de,page_1,6.2,11751,0.49,22,commercial
4,content_d99b7a2d90ca,client_3fdba35f04,page_3_5,44.0,19140,0.13,14,informational


## 7. Sketching the target column

There's no single "target" column to predict — the score is computed, not learned. The cell below
builds it step by step so the shape of the eventual queue is visible:

1. `expected_ctr` — the position tier's median CTR (the leakage-safe baseline)
2. `gap_score` — how far below that baseline this page sits
3. `freshness_penalty` — bucketed off the dataset's own `days_since_last_update`, so a
   recently-refreshed page's rough CTR isn't mistaken for a genuine content problem
4. `priority_score` — the actual ranking column, `gap_score × log(impressions) × (1 + freshness_penalty)`

Only pages with a **positive** gap (genuinely below their tier's typical CTR) are review
candidates at all — a page above its tier's median isn't a "reverse" problem, so those get scored
at 0, not a negative priority.

In [3]:
import numpy as np

# Step 1-2: tier baseline + gap
visible["expected_ctr"] = visible.groupby("position_tier")["ctr"].transform("median")
visible["gap_score"] = (visible["expected_ctr"] - visible["ctr"]).clip(lower=0)

# Step 3: freshness -- bucketed off the dataset's own days_since_last_update column
visible["freshness_penalty"] = pd.cut(
    visible["days_since_last_update"], bins=[-1, 30, 180, 10_000], labels=[0.0, 0.2, 0.5]
).astype(float)

# Step 4: the actual ranking column
visible["priority_score"] = (
    visible["gap_score"] * np.log1p(visible["impressions_90d"]) * (1 + visible["freshness_penalty"])
)

queue = (visible[visible["gap_score"] > 0]
         .sort_values("priority_score", ascending=False)
         [["content_id", "client_id", "position_tier", "avg_position", "impressions_90d",
           "ctr", "expected_ctr", "gap_score", "days_since_last_update", "priority_score"]])

print(f"review-queue candidates (gap_score > 0): {len(queue):,} of {len(visible):,} visible pages\n")
queue.head(10)


review-queue candidates (gap_score > 0): 7,950 of 16,726 visible pages



,content_id,client_id,position_tier,avg_position,impressions_90d,ctr,expected_ctr,gap_score,days_since_last_update,priority_score
7445,content_c8e9d6ab9013,client_19581e27de,page_1,9.7,208678,0.00,0.24,0.24,104,3.527583
9193,content_c1fe78bc4e37,client_19581e27de,page_1,7.5,134055,0.03,0.24,0.21,104,2.975115
4708,content_b115f7c74779,client_19581e27de,page_1,8.0,123469,0.03,0.24,0.21,104,2.954386
3394,content_36ff89c8214e,client_19581e27de,page_1,7.3,295097,0.05,0.24,0.19,104,2.871674
10646,content_d0cc5baa4995,client_19581e27de,page_1,6.6,83651,0.03,0.24,0.21,104,2.856274
25462,content_825a9788af8d,client_4e07408562,page_1,5.6,16786,0.00,0.24,0.24,104,2.801768
9443,content_8ba781dafa55,client_8527a891e2,page_1,9.0,16156,0.00,0.24,0.24,104,2.790751
4895,content_d07ea098353c,client_19581e27de,page_1,9.4,63366,0.03,0.24,0.21,104,2.786288
18460,content_a38dd531fd8f,client_19581e27de,page_1,6.5,22716,0.01,0.24,0.23,104,2.768520
7785,content_c9eeaab4031e,client_19581e27de,page_1,6.2,34100,0.02,0.24,0.22,104,2.755390
